In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [3]:
with Image.open("./datasets/Training/glioma/Tr-gl_10.jpg") as im:
    print(im.width, im.height)

512 512


In [4]:
TRAINING_DIR = Path('./datasets/Training')
TESTING_DIR = Path('./datasets/Testing')

transform = v2.Compose([
    v2.Resize(size=(512, 512)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True), #ToTensor
])

train_dataset = datasets.ImageFolder(root=TRAINING_DIR, transform=transform)
test_dataset = datasets.ImageFolder(root=TESTING_DIR, transform=transform)

print(f"Classes: {train_dataset.classes}")
print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples:  {len(test_dataset)}")

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Training samples: 5600
Testing samples:  1600


In [5]:
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches:  {len(test_loader)}")

# Verify a batch
images, labels = next(iter(train_loader))
print(f"Batch image shape: {images.shape}")   # (32, 3, 512, 512)
print(f"Batch label shape: {labels.shape}")   # (32,)
print(f"Label mapping: {train_dataset.class_to_idx}")

Train batches: 175
Test batches:  50
Batch image shape: torch.Size([32, 3, 512, 512])
Batch label shape: torch.Size([32])
Label mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

class BrainTumorCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3,   32),   # 512 -> 256
            conv_block(32,  64),   # 256 -> 128
            conv_block(64,  128),  # 128 ->  64
            conv_block(128, 256),  #  64 ->  32
            conv_block(256, 512),  #  32 ->  16
            conv_block(512, 512),  #  16 ->   8
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),   # 8 -> 4  (512*4*4 = 8192)
            nn.Flatten(),
            nn.Linear(512 * 4 * 4, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = BrainTumorCNN(num_classes=4).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

Using device: cuda
BrainTumorCNN(
  (features): Sequential(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3

In [ ]:
EPOCHS = 20
LR = 1e-3
CHECKPOINT = Path("best_model.pth")

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

history = {"train_loss": [], "train_acc": [], "test_loss": [], "test_acc": []}
best_acc = 0.0

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    train_loss = running_loss / total
    train_acc  = correct / total

    # --- Evaluation ---
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    test_loss = running_loss / total
    test_acc  = correct / total

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["test_loss"].append(test_loss)
    history["test_acc"].append(test_acc)

    # --- Save best checkpoint ---
    saved = ""
    if test_acc > best_acc:
        best_acc = test_acc
        torch.save(model.state_dict(), CHECKPOINT)
        saved = "  *** saved ***"

    print(f"Epoch {epoch+1:02d}/{EPOCHS}  "
          f"train loss: {train_loss:.4f}  train acc: {train_acc:.4f}  "
          f"test loss: {test_loss:.4f}  test acc: {test_acc:.4f}{saved}")

print(f"\nBest test accuracy: {best_acc:.4f}  →  saved to '{CHECKPOINT}'")

Epoch 01/20  train loss: 0.9949  train acc: 0.6559  test loss: 1.0232  test acc: 0.6481  *** saved ***


In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], label="train")
ax1.plot(epochs, history["test_loss"],  label="test")
ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

ax2.plot(epochs, history["train_acc"], label="train")
ax2.plot(epochs, history["test_acc"],  label="test")
ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend()

plt.tight_layout()
plt.show()

print(f"Best test accuracy: {max(history['test_acc']):.4f}")

: 

: 

: 